# Clouddex — train the cloud classifier on Google Colab (free GPU)

This notebook trains the MobileNetV2 cloud-genus classifier and exports it to
**TensorFlow.js** for the Clouddex PWA — entirely in the cloud, no local install.

**How to use**
1. Runtime → **Change runtime type** → Hardware accelerator = **GPU** → Save.
2. Run the cells top to bottom.
3. Provide the **CCSN** dataset when prompted (upload a `.zip`, mount Drive, or a URL).
4. The last cell downloads `clouddex_model.zip`. Unzip it into the repo's
   `public/model/` folder, commit, and push — the live site redeploys with the
   real model.

> Preprocessing here (224×224, MobileNetV2 `[-1,1]`) matches `src/ml/preprocess.ts`
> in the app. Don't change one without the other.

In [ ]:
# 1. Sanity check: TensorFlow + GPU
import tensorflow as tf
print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE — set Runtime > Change runtime type > GPU")

## 2. Get the CCSN dataset

Pick **one** option below. Each class must end up as its own folder under
`DATA_DIR` (either CCSN's 2-letter codes `Ci/ Cu/ …` or full names — the next
cell normalizes them).

In [ ]:
# --- Option A (default): upload a .zip of the dataset ---
import os, zipfile
from google.colab import files

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()  # choose your CCSN .zip
for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(DATA_DIR)
        print("extracted", name)

# If everything nested under a single top folder, descend into it.
subdirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
if len(subdirs) == 1:
    inner = os.path.join(DATA_DIR, subdirs[0])
    if any(os.path.isdir(os.path.join(inner, d)) for d in os.listdir(inner)):
        DATA_DIR = inner
print("DATA_DIR =", DATA_DIR)
print("classes:", sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))))

In [ ]:
# --- Option B: mount Google Drive (put the dataset in your Drive first) ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/CCSN"   # adjust to your folder

# --- Option C: download from a direct URL (a .zip) ---
# import os, zipfile, urllib.request
# DATA_DIR = "/content/data"; os.makedirs(DATA_DIR, exist_ok=True)
# urllib.request.urlretrieve("https://example.com/CCSN.zip", "/content/ccsn.zip")
# with zipfile.ZipFile("/content/ccsn.zip") as z: z.extractall(DATA_DIR)

In [ ]:
# 3. Normalize CCSN 2-letter folder codes -> full class ids the app expects
import os
MAP = {"Ci": "cirrus", "Cs": "cirrostratus", "Cc": "cirrocumulus",
       "Ac": "altocumulus", "As": "altostratus", "Cu": "cumulus",
       "Cb": "cumulonimbus", "Ns": "nimbostratus", "Sc": "stratocumulus",
       "St": "stratus", "Ct": "contrail"}
for code_, full in MAP.items():
    src, dst = os.path.join(DATA_DIR, code_), os.path.join(DATA_DIR, full)
    if os.path.isdir(src) and not os.path.isdir(dst):
        os.rename(src, dst); print(f"renamed {code_} -> {full}")
print("final classes:",
      sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))))

In [ ]:
# 4. Build train/val datasets + class weights (CCSN is imbalanced)
import os, tensorflow as tf
IMG_SIZE, BATCH = 224, 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="categorical")
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="categorical")

class_names = train_ds.class_names
print("MODEL OUTPUT ORDER (this becomes labels.json):", class_names)

counts = [len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names]
total, n = sum(counts), len(class_names)
class_weight = {i: (total / (n * c) if c else 1.0) for i, c in enumerate(counts)}

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# 5. MobileNetV2 transfer-learning model
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

augment = models.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
    layers.RandomZoom(0.1), layers.RandomContrast(0.1)], name="augment")

base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                   weights="imagenet")
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = preprocess_input(x)          # -> [-1, 1], matches src/ml/preprocess.ts
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# 6. Train the head (feature extraction)
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight)

In [ ]:
# 7. Fine-tune the top of the backbone at a low LR
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight)

In [ ]:
# 8. Export a SavedModel + labels.json (in the model's true output order)
import json
model.export("/content/saved_model")
with open("/content/labels.json", "w") as f:
    json.dump({"labels": class_names}, f, indent=2)
print("Saved /content/saved_model and labels:", class_names)

## 9. Convert to TensorFlow.js

Installs `tensorflowjs` and runs the converter as a subprocess (so any TF
version change pip makes doesn't disturb this kernel).

> If the converter errors on a version mismatch, just **Runtime → Restart**, then
> run this cell again (your `/content/saved_model` is already on disk). As a
> fallback you can download the SavedModel and convert locally with the `convert.sh`
> in the repo — your local env already has the converter.

In [ ]:
# 9. Convert SavedModel -> TF.js graph model
import os, shutil
os.system("pip install -q tensorflowjs")
os.makedirs("/content/web_model", exist_ok=True)
rc = os.system(
    "tensorflowjs_converter --input_format=tf_saved_model "
    "--output_format=tfjs_graph_model --signature_name=serving_default "
    "--saved_model_tags=serve /content/saved_model /content/web_model")
assert rc == 0, "conversion failed — see the note above this cell"
shutil.copy("/content/labels.json", "/content/web_model/labels.json")
print("web_model contents:", os.listdir("/content/web_model"))

In [ ]:
# 10. Zip the model and download it
import shutil
from google.colab import files
shutil.make_archive("/content/clouddex_model", "zip", "/content/web_model")
files.download("/content/clouddex_model.zip")

## 11. Install the model in the app

1. Unzip `clouddex_model.zip` — you'll get `model.json`, one or more `*.bin`
   weight shards, and `labels.json`.
2. Put all of those into the repo's **`public/model/`** folder (replacing the
   placeholder `labels.json`).
3. Commit and push:
   ```bash
   git add public/model && git commit -m "Add trained model" && git push
   ```
4. The GitHub Actions deploy runs automatically; the live site loses its
   **“demo model”** badge and starts making real predictions.